# Plant NeRF — COLMAP poses (train, validate, export, save)

Trains `nerfacto` on the images using **COLMAP** SfM poses, validates the model,
exports multiple output formats (point cloud / mesh / video), and saves everything
to Drive.

**Before running:** upload `nerf_ready_dataset.zip` (built by the terminal command) to your
Drive, e.g. `MyDrive/plant_pheno/nerf_ready_dataset.zip`, and set `ZIP_PATH` below.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 1 — Mount Drive, set paths

import os

ZIP_PATH   = "/content/drive/MyDrive/jetcobot_2026/nerf/nerf_ready_dataset.zip"  # <-- edit if needed
WORK       = "/content/work"
DRIVE_OUT  = "/content/drive/MyDrive/jetcobot_2026/nerf/output"      # everything final gets copied here

os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)
print("Zip source:", ZIP_PATH)
print("Drive output folder:", DRIVE_OUT)


Zip source: /content/drive/MyDrive/jetcobot_2026/nerf/nerf_ready_dataset.zip
Drive output folder: /content/drive/MyDrive/jetcobot_2026/nerf/output


In [1]:
# CELL 2 — Install nerfstudio + deps (GPU runtime required: Runtime > Change runtime type > GPU)
!pip install -q --upgrade pip
!pip install -q nerfstudio open3d trimesh
!ns-install-cli || true
print("nerfstudio install done")


[08:22:31] 🤷 .zshrc not found, skipping.                                                                 ]8;id=102480;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/completions/install.py\install.py]8;;\:]8;id=776185;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/completions/install.py#369\369]8;;\
           🔍 Found .bashrc!                                                                              ]8;id=261338;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/completions/install.py\install.py]8;;\:]8;id=529404;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/completions/install.py#371\371]8;;\
[08:22:33] ✔ Nothing to do for                                                                            ]8;id=366398;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/completions/install.py\install.py]8;;\:]8;id=671540;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/scripts/

In [4]:
# CELL 3 — Unzip the combined dataset
import zipfile, shutil, glob

extract_dir = os.path.join(WORK, "raw")
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(extract_dir)

# handle the case where zip contains a top-level 'nerf_ready_dataset' folder
root_candidates = glob.glob(os.path.join(extract_dir, '*'))
DATA_ROOT = extract_dir if os.path.exists(os.path.join(extract_dir, 'images')) else root_candidates[0]
print("DATA_ROOT:", DATA_ROOT)
print(os.listdir(DATA_ROOT))


DATA_ROOT: /content/work/raw/nerf_ready_dataset
['images', 'colmap', 'mast3r']


In [5]:
# CELL 4 — Build the COLMAP-based nerfstudio dataset folder
# nerfstudio's colmap dataparser expects: <dataset>/images/*  and  <dataset>/colmap/sparse/0/*
colmap_ds = os.path.join(WORK, "dataset_colmap")
os.makedirs(os.path.join(colmap_ds, "images"), exist_ok=True)
os.makedirs(os.path.join(colmap_ds, "colmap", "sparse", "0"), exist_ok=True)

shutil.copytree(os.path.join(DATA_ROOT, "images"), os.path.join(colmap_ds, "images"), dirs_exist_ok=True)
shutil.copytree(os.path.join(DATA_ROOT, "colmap", "sparse", "0"),
                 os.path.join(colmap_ds, "colmap", "sparse", "0"), dirs_exist_ok=True)

print("COLMAP dataset ready at:", colmap_ds)
!ls {colmap_ds}/colmap/sparse/0


COLMAP dataset ready at: /content/work/dataset_colmap
cameras.bin  images.bin  points3D.bin  project.ini


In [6]:
# CELL 5 — Train NeRF: COLMAP poses
!ns-train nerfacto \
  --data {colmap_ds} \
  --output-dir {WORK}/outputs \
  --experiment-name colmap_run \
  --viewer.quit-on-train-completion True \
  --max-num-iterations 10000 \
  colmap


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
6400 (64.00%)       303.226 ms           18 m, 11 s           14.18 K                                
6410 (64.10%)       309.254 ms           18 m, 30 s           14.06 K                                
6420 (64.20%)       290.960 ms           17 m, 21 s           14.75 K                                
6430 (64.30%)       291.713 ms           17 m, 21 s           14.73 K                                
6440 (64.40%)       310.303 ms           18 m, 24 s           13.97 K                                
---------------------------------------------------------------------------------------------------- 
Viewer running locally at: http://localhost:7007 (listening on 0.0.0.0)                              
Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec                       
-----------------------------------------------------------------------------------                  
6360

In [7]:
# CELL 6 — Locate the trained config
import glob

def latest_config(exp_name):
    matches = sorted(glob.glob(f"{WORK}/outputs/{exp_name}/nerfacto/*/config.yml"))
    assert matches, f"no config found for {exp_name}"
    return matches[-1]

colmap_cfg = latest_config("colmap_run")
print("COLMAP config:", colmap_cfg)


COLMAP config: /content/work/outputs/colmap_run/nerfacto/2026-07-24_082441/config.yml


In [23]:
# CELL 8 — Export multiple output types (point cloud, poisson mesh, TSDF mesh) + turntable video
export_dir = os.path.join(WORK, "exports")

for out_type in ["pointcloud", "poisson", "tsdf"]:
    out_path = os.path.join(export_dir, "colmap", out_type)
    os.makedirs(out_path, exist_ok=True)
    print(f"--- exporting {out_type} for colmap ---")
    !ns-export {out_type} --load-config {colmap_cfg} --output-dir {out_path}

# also render a turntable flythrough video, useful for visual comparison
render_path = os.path.join(export_dir, "colmap", "render.mp4")
!ns-render spiral --load-config {colmap_cfg} --output-path {render_path} || echo "render skipped, define a camera path if this fails"


--- exporting pointcloud for colmap ---
2026-07-24 09:41:42.248382: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[09:41:49] Using image downscale factor of 1                                                    ]8;id=746560;file:///usr/local/lib/p

In [24]:
import os

for root, dirs, files in os.walk(export_dir):
    level = root.replace(export_dir, "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")
    for f in files:
        print(indent + "  " + f)

exports/
  colmap/
    pointcloud/
    poisson/
    tsdf/


In [27]:
import pathlib

path = "/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py"

text = pathlib.Path(path).read_text()

text = text.replace(
    "torch.load(load_path, map_location=\"cpu\")",
    "torch.load(load_path, map_location=\"cpu\", weights_only=False)"
)

pathlib.Path(path).write_text(text)

print("patched")

patched


In [35]:
!ns-export pointcloud \
--load-config {colmap_cfg} \
--output-dir {export_dir}/colmap/pointcloud \
--normal-method open3d

2026-07-24 09:56:01.721992: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[09:56:10] Using image downscale factor of 1                                                    ]8;id=666188;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/

In [36]:
!ls -lh {export_dir}/colmap/pointcloud

total 26M
-rw-r--r-- 1 root root 26M Jul 24 09:57 point_cloud.ply


In [37]:
eval_dir = os.path.join(WORK, "eval")
os.makedirs(eval_dir, exist_ok=True)

colmap_metrics_path = os.path.join(eval_dir, "colmap_metrics.json")

!ns-eval \
--load-config {colmap_cfg} \
--output-path {colmap_metrics_path}

2026-07-24 09:58:40.916011: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[09:58:46] Using image downscale factor of 1                                                    ]8;id=469000;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/

In [38]:
import json

with open(colmap_metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "experiment_name": "colmap_run",
  "method_name": "nerfacto",
  "checkpoint": "/content/work/outputs/colmap_run/nerfacto/2026-07-24_082441/nerfstudio_models/step-000009999.ckpt",
  "results": {
    "psnr": 19.929489135742188,
    "psnr_std": 4.6852264404296875,
    "ssim": 0.7906454205513,
    "ssim_std": 0.07555194199085236,
    "lpips": 0.16015847027301788,
    "lpips_std": 0.18360859155654907,
    "num_rays_per_sec": 29417.68359375,
    "num_rays_per_sec_std": 586.3195190429688,
    "fps": 0.09576069563627243,
    "fps_std": 0.001908592996187508
  }
}


In [39]:
render_path = os.path.join(export_dir, "colmap", "spiral.mp4")

!ns-render spiral \
--load-config {colmap_cfg} \
--output-path {render_path}

2026-07-24 10:03:23.830447: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[10:03:30] Using image downscale factor of 1                                                    ]8;id=722401;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/

In [40]:
!ls -lh {render_path}

-rw-r--r-- 1 root root 955K Jul 24 10:16 /content/work/exports/colmap/spiral.mp4


In [41]:
tsdf_dir = os.path.join(export_dir, "colmap", "tsdf")

!ns-export tsdf \
--load-config {colmap_cfg} \
--output-dir {tsdf_dir}

2026-07-24 10:16:34.749199: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[10:16:43] Using image downscale factor of 1                                                    ]8;id=150726;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/

In [42]:
poisson_dir = os.path.join(export_dir, "colmap", "poisson")

!ns-export poisson \
--load-config {colmap_cfg} \
--output-dir {poisson_dir} \
--normal-method open3d

2026-07-24 10:28:53.015498: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[10:29:00] Using image downscale factor of 1                                                    ]8;id=901738;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/data/

## Notes
- **Re-opening later in nerfstudio**: point `ns-viewer --load-config <checkpoint/config.yml>` at the saved checkpoint folder to interactively re-view the trained model.
- **CloudCompare / MeshLab / Blender**: open any `.ply` under `exports/pointcloud|poisson|tsdf/`.
- `--max-num-iterations 30000` is a reasonable default; drop to 10000–15000 first if you just want a fast sanity check before committing to a full run.
- The printed `final_dir` path above is what you paste into the comparison notebook.
